# E7 step 2c · Option (c): a mix of SAM 3's outlines, the grader and R11, chosen on validation (PC, no GPU)

Task **E7**, step 2c. I built and ran it on 17 Nov 2026. It trains nothing and runs no model: everything comes from files the earlier runs saved.

**The question.** On the 2,007 unseen test pairs (E7 step 2), find-then-grade and R11 were level on destroyed buildings (0.3898 against 0.3984), but each won where the other lost:
- outside Turkey, R11 led by 0.057;
- on Turkey, find-then-grade led by 0.236 (E7 step 2b).

Can a mix of the two beat both?

**The two methods being mixed**
- **R11:** our U-Net that reads the after-photo and the before-photo together, with 8-view TTA. It labels every pixel.
- **Find-then-grade:** SAM 3 outlines each building on the before-photo. Our grader, a Siamese ResNet-18 with three seeds (42, 43, 44), labels each outline intact, damaged, destroyed or "not a building".

**What's saved, for every SAM 3 outline:**
- the three graders' log-probabilities;
- R11's pixels of each class under the outline;
- the true pixels of each class under it.

These are R16c steps 5 and 5b's `instances_val.csv` for validation, and E7 step 2's `instances_test.csv` for test. A whole map's score also needs the true pixels outside the outlines. Those come from E5 step 2's per-image counts (validation) and E6's masks (test).

**Check first (the notebook stops if these fail).** Rebuilt from those files, the scores must reproduce the published ones within 0.0005:
- find-then-grade per seed on the 605 clean validation pairs: destroyed IoU 0.3161, 0.3051 and 0.2982 (R16c steps 5 and 5b);
- on the clean test pairs: seed 42's 0.4055 and mIoU 0.5179;
- R11's 0.3984 and 0.5721 (E7 step 2's `map_scores.csv`).

**Amended after the first run (17 Nov), which stopped at this check.**
- Seed 43 on validation rebuilt to 0.3063, against 0.3051 published.
- R16c step 5 recorded on 26 Oct that seed 43's per-building file rebuilds its maps only to within 0.0012: its saved log-probabilities are rounded, so near-ties can flip. Its published clean score used the lower value.
- So that one check allows 0.0015, and every other check stays at 0.0005. All other checks passed in the first run. E7 step 2b handled the same rounding the same way.
- Nothing else changed. The candidates are all rebuilt from the same files, so they compare like with like.

**The candidates (fixed before the run).** Each one gives every SAM 3 outline a class, and everything outside the outlines is background, as in find-then-grade; candidate C4 is the exception.
- **C1, R11 grades the outlines:** each outline takes R11's most common class among its pixels, background included.
- **C2, R11 grades, the grader decides "is it a building":** an outline the grader calls "not a building" is background; any other takes R11's most common building class under it. If R11 has no building pixel there, the grader's own class is kept.
- **C3, a blend:** w × the grader's probabilities + (1 − w) × R11's class shares under the outline, with "not a building" paired with R11's background; w ∈ {0.25, 0.5, 0.75}.
- **C4, by image source:** R11's whole map on xBD images, and find-then-grade on the others (Turkey and the rest of BRIGHT's events).
  - DisasterM3's BRIGHT tiles are 0.30 m per pixel, against xBD's 1.25–2.82 m, so a real system could switch by ground resolution.
  - Here the source comes from the file name (E1's rule).
- **References:** R11 alone; find-then-grade alone.

Every candidate that uses the grader is scored with each seed separately and reported as the mean of the three, as E7 did.

**The rule that picks one (fixed before the run).**
- The candidate with the highest destroyed IoU on the 605 clean validation pairs wins.
- A tie within 0.002 goes to the simpler one, in the order C1, C4, C2, C3.
- The winner is then scored once on the 2,007 clean test pairs, beside R11 and find-then-grade, with nothing changed. The pairs are redrawn 10,000 times, as in E7 step 2b, to show whether its lead over each is more than noise.
- **This notebook doesn't choose the system's method.** It gives me the evidence for option (c).

**Expectation, fixed before the run (rough guesses, not predictions):**
- the rebuilt scores reproduce the published ones;
- on validation, C4 is likely the strongest (R11 led on xBD there too; find-then-grade on Turkey);
- C1 lands near R11's 0.27, and C3 near the grader's 0.31;
- if C4 wins, on test it may reach about 0.43–0.45, above both 0.3984 and 0.3898. But Turkey is 20% of the test pairs against 10% of validation, so the gain could differ.

**Outputs** in `E7-results/step2c/`:
- `README.md`;
- `candidates_validation.csv`;
- `test_confirmation.csv`;
- `resampling.csv`.

In [1]:
# ── Cell 1: Inputs and helpers (run from Essential_task_delegation/Champion_Task) ──
import io
import json
import re
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

OUT = Path("E7-results/step2c")
OUT.mkdir(parents=True, exist_ok=True)
LABELS = ["intact", "damaged", "destroyed", "none"]        # the grader's labels, in its output order
SEEDS = (42, 43, 44)
R11_COLS = ["r11_background_px", "r11_intact_px", "r11_damaged_px", "r11_destroyed_px"]
GT_COLS = ["gt_background_px", "gt_intact_px", "gt_damaged_px", "gt_destroyed_px"]
N_RESAMPLES, RNG_SEED = 10_000, 0                          # as E7 step 2b
TIE = 0.002
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")


def event_of(name):
    """E1's rule: 'hurricane_florence_00000261_post_disaster.png' -> 'hurricane_florence'."""
    m = XBD_NAME.match(Path(name).stem)
    if m:
        return m.group(1)
    m = OTHER_NAME.match(Path(name).stem)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem)


def source_of(name):
    return "xBD" if XBD_NAME.match(Path(name).stem) else "other"


# The graders' label biases, from their calibration files (R16c steps 5 and 5b)
BIAS = {}
for path in ("R16c-results/step5/calibration.json", "R16c-results/step5b/calibration.json"):
    for seed, chosen in json.loads(Path(path).read_text(encoding="utf-8")).items():
        BIAS[int(seed)] = np.array([chosen["bias"][l.capitalize() if l != "none" else "None"] for l in LABELS])
assert sorted(BIAS) == list(SEEDS), BIAS

# ── Validation: the 605 clean pairs ──
leaks = pd.read_csv("E5-results/step1/val_train_leaks_hash.csv")
LEAKED = set(leaks.loc[leaks["counted"].astype(str) == "True", "val_image"])
pc = pd.read_csv("E5-results/step2/per_image_counts.csv")
r11v = pc[(pc["model"] == "R11") & (pc["mode"] == "tta")].set_index("image")
CM_COLS = [f"t{t}_p{p}" for t in range(4) for p in range(4)]
VAL_IMAGES = sorted(set(r11v.index) - LEAKED)
R11_VAL = {i: r11v.loc[i, CM_COLS].to_numpy(np.int64).reshape(4, 4) for i in VAL_IMAGES}
GT_VAL = {i: cm.sum(1) for i, cm in R11_VAL.items()}         # true pixels per class, per image
iv5 = pd.read_csv("R16c-results/step5/instances_val.csv").query("found_by == 'sam3'")
iv5b = pd.read_csv("R16c-results/step5b/instances_val.csv").query("found_by == 'sam3'")
IV = iv5.merge(iv5b[["image", "id"] + [f"logp_{l}_s44" for l in LABELS]], on=["image", "id"], how="left", validate="one_to_one")
assert IV[[f"logp_{l}_s44" for l in LABELS]].notna().all().all(), "seed 44's grades don't cover every outline"
IV = IV[IV["image"].isin(VAL_IMAGES)].reset_index(drop=True)

# ── Test: the 2,007 clean pairs (E6), with R11's saved maps and the true masks ──
inputs = zipfile.ZipFile("E7-inputs/e7-inputs.zip")
pairs = pd.read_csv(io.BytesIO(inputs.read("test_pairs.csv")))
TEST_IMAGES = sorted(pairs.loc[pairs["clean"].astype(str) == "True", "after_image"])
masks = zipfile.ZipFile(io.BytesIO(inputs.read("test_masks_4class.zip")))
r11_maps = zipfile.ZipFile("E7-results/step2/r11_tta_maps.zip")
GT_TEST, R11_TEST = {}, {}
for name in TEST_IMAGES:
    pred = cv2.imdecode(np.frombuffer(r11_maps.read(name), np.uint8), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imdecode(np.frombuffer(masks.read(name), np.uint8), cv2.IMREAD_GRAYSCALE)
    if gt.shape != pred.shape:                               # E7 read each mask at its image's size
        gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST)
    GT_TEST[name] = np.bincount(gt.ravel(), minlength=4)[:4]
    R11_TEST[name] = np.bincount(gt.ravel().astype(np.int64) * 4 + pred.ravel(), minlength=16).reshape(4, 4)
IT = pd.read_csv("E7-results/step2/instances_test.csv")
IT = IT[IT["image"].isin(TEST_IMAGES)].reset_index(drop=True)


# ── Helpers ──
def ftg_classes(df, seed):
    """Find-then-grade: each outline's class from the grader (with its biases): 1-3, or 0 for "not a building"."""
    label = (df[[f"logp_{l}_s{seed}" for l in LABELS]].to_numpy(np.float64) + BIAS[seed]).argmax(1)
    return np.where(label < 3, label + 1, 0)


def grader_probs(df, seed):
    """The grader's calibrated probabilities (with its biases), columns intact, damaged, destroyed, none."""
    z = df[[f"logp_{l}_s{seed}" for l in LABELS]].to_numpy(np.float64) + BIAS[seed]
    z -= z.max(1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(1, keepdims=True)


def per_image_cms(df, cls, images, gt_totals):
    """Each image's 4x4 confusion (rows truth, columns prediction) for a map made of the outlines with these classes,
    background elsewhere: the true pixels under each outline go to its class, the rest of the image to background."""
    out = {i: np.zeros((4, 4), np.int64) for i in images}
    gt = df[GT_COLS].to_numpy(np.int64)
    for img, idx in df.groupby("image").indices.items():
        if img in out:
            for c in range(4):
                out[img][:, c] += gt[idx][cls[idx] == c].sum(0)
    for img in images:
        out[img][:, 0] += gt_totals[img] - out[img].sum(1)
    return out


def scores(cm):
    tp = np.diag(cm).astype(np.float64)
    iou = tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)
    b_tp = cm[1:, 1:].sum()
    return {"miou": iou.mean(), "destroyed_iou": iou[3], "building_iou": b_tp / max(b_tp + cm[0, 1:].sum() + cm[1:, 0].sum(), 1)}


def total(cms, images):
    return sum((cms[i] for i in images), np.zeros((4, 4), np.int64))


# Self-checks of the helpers on a made-up image: one outline of 10 true destroyed px graded destroyed, 5 true intact px outside
_df = pd.DataFrame({"image": ["a"], **{c: [v] for c, v in zip(GT_COLS, [0, 0, 0, 10])}})
_cm = per_image_cms(_df, np.array([3]), ["a"], {"a": np.array([100, 5, 0, 10])})["a"]
assert _cm[3, 3] == 10 and _cm[1, 0] == 5 and _cm[0, 0] == 100 and _cm.sum() == 115
assert abs(scores(_cm)["destroyed_iou"] - 1) < 1e-12 and source_of("turkey_earthquake0_post_380.png") == "other"
print(f"✓ Validation: {len(VAL_IMAGES)} clean pairs, {len(IV):,} SAM 3 outlines on {IV['image'].nunique()} of them")
print(f"✓ Test: {len(TEST_IMAGES):,} clean pairs, {len(IT):,} SAM 3 outlines on {IT['image'].nunique():,} of them; biases {dict((s, BIAS[s].tolist()) for s in SEEDS)}")

✓ Validation: 605 clean pairs, 26,175 SAM 3 outlines on 539 of them
✓ Test: 2,007 clean pairs, 137,760 SAM 3 outlines on 1,887 of them; biases {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}


In [2]:
# ── Cell 2: Check first: the rebuilt scores must reproduce the published ones (within 0.0005) ──
published = pd.read_csv("E7-results/step2/map_scores.csv")
pub = published[published["scope"] == "clean pairs"].set_index("method")
checks = []


def check(what, mine, theirs, tol=0.0005):
    checks.append({"check": what, "rebuilt": round(float(mine), 4), "published": round(float(theirs), 4),
                   "pass": abs(mine - theirs) <= tol})


# Seed 43 on validation is allowed 0.0015 (amended after the first run, 17 Nov, which stopped here: 0.3063 rebuilt against
# 0.3051 published). R16c step 5 recorded on 26 Oct that seed 43's per-building file rebuilds its maps only to within
# 0.0012: its saved log-probabilities are rounded, so near-ties can flip. Its published clean score used the lower value.
# Every other check stays at 0.0005. All candidates are rebuilt from the same files, so they compare like with like.
for seed, published_destroyed in zip(SEEDS, (0.3161, 0.3051, 0.2982)):          # R16c steps 5 and 5b, clean pairs
    s = scores(total(per_image_cms(IV, ftg_classes(IV, seed), VAL_IMAGES, GT_VAL), VAL_IMAGES))
    check(f"validation, find-then-grade seed {seed}: destroyed IoU", s["destroyed_iou"], published_destroyed,
          tol=0.0015 if seed == 43 else 0.0005)
s = scores(total(R11_VAL, VAL_IMAGES))
check("validation, R11 with TTA: destroyed IoU", s["destroyed_iou"], 0.2692)
check("validation, R11 with TTA: mIoU", s["miou"], 0.5805)
for seed in SEEDS:
    s = scores(total(per_image_cms(IT, ftg_classes(IT, seed), TEST_IMAGES, GT_TEST), TEST_IMAGES))
    check(f"test, find-then-grade seed {seed}: destroyed IoU", s["destroyed_iou"], pub.loc[f"chain_s{seed}", "destroyed_iou"])
    check(f"test, find-then-grade seed {seed}: mIoU", s["miou"], pub.loc[f"chain_s{seed}", "miou"])
s = scores(total(R11_TEST, TEST_IMAGES))
check("test, R11 with TTA: destroyed IoU", s["destroyed_iou"], pub.loc["R11_tta", "destroyed_iou"])
check("test, R11 with TTA: mIoU", s["miou"], pub.loc["R11_tta", "miou"])
CHECKS = pd.DataFrame(checks)
print(CHECKS.to_string(index=False))
if not CHECKS["pass"].all():
    raise RuntimeError("The rebuilt scores don't reproduce the published ones, so nothing below can be trusted. Stop.")
print("✓ Every rebuilt score reproduces its published value")

                                             check  rebuilt  published  pass
validation, find-then-grade seed 42: destroyed IoU   0.3161     0.3161  True
validation, find-then-grade seed 43: destroyed IoU   0.3063     0.3051  True
validation, find-then-grade seed 44: destroyed IoU   0.2982     0.2982  True
           validation, R11 with TTA: destroyed IoU   0.2692     0.2692  True
                    validation, R11 with TTA: mIoU   0.5805     0.5805  True
      test, find-then-grade seed 42: destroyed IoU   0.4055     0.4055  True
               test, find-then-grade seed 42: mIoU   0.5180     0.5179  True
      test, find-then-grade seed 43: destroyed IoU   0.3982     0.3983  True
               test, find-then-grade seed 43: mIoU   0.5121     0.5121  True
      test, find-then-grade seed 44: destroyed IoU   0.3657     0.3657  True
               test, find-then-grade seed 44: mIoU   0.5086     0.5087  True
                 test, R11 with TTA: destroyed IoU   0.3984     0.3984  True

In [3]:
# ── Cell 3: The candidates on the 605 clean validation pairs, and the rule's choice ──
def r11_shares(df):
    """R11's pixels of each class under each outline, as shares: columns background, intact, damaged, destroyed."""
    px = df[R11_COLS].to_numpy(np.float64)
    return px / np.maximum(px.sum(1, keepdims=True), 1)


def c1(df, seed):                       # R11 grades the outlines: its most common class, background included
    return df[R11_COLS].to_numpy().argmax(1)


def c2(df, seed):                       # the grader says whether it's a building; R11 says which building class
    g = ftg_classes(df, seed)
    b = df[R11_COLS[1:]].to_numpy()
    return np.where(g == 0, 0, np.where(b.sum(1) > 0, b.argmax(1) + 1, g))


def c3(w):
    def rule(df, seed):                 # w x grader + (1 - w) x R11's shares; "none" paired with background
        s = r11_shares(df)
        mix = w * grader_probs(df, seed) + (1 - w) * np.column_stack([s[:, 1], s[:, 2], s[:, 3], s[:, 0]])
        label = mix.argmax(1)
        return np.where(label < 3, label + 1, 0)
    return rule


def map_cms(rule, df, images, gt_totals, r11_cms, seed):
    """Each image's confusion for a candidate; "c4" uses R11's whole map on xBD images and find-then-grade elsewhere."""
    if rule == "c4":
        ftg = per_image_cms(df, ftg_classes(df, seed), images, gt_totals)
        return {i: (r11_cms[i] if source_of(i) == "xBD" else ftg[i]) for i in images}
    if rule == "r11":
        return {i: r11_cms[i] for i in images}
    if rule == "ftg":
        return per_image_cms(df, ftg_classes(df, seed), images, gt_totals)
    return per_image_cms(df, rule(df, seed), images, gt_totals)


CANDIDATES = {"C1, R11 grades the outlines": c1, "C2, grader: building or not; R11: which class": c2,
              "C3, blend, w = 0.25": c3(0.25), "C3, blend, w = 0.5": c3(0.5), "C3, blend, w = 0.75": c3(0.75),
              "C4, R11 on xBD images, find-then-grade on the others": "c4"}
REFERENCES = {"R11 alone (reference)": "r11", "Find-then-grade alone (reference)": "ftg"}
ORDER = ["C1", "C4", "C2", "C3"]          # the tie-break: the simpler candidate first


def evaluate(rule, df, images, gt_totals, r11_cms):
    """Scores per seed and their mean (R11 alone and C1 don't use the grader: their three seeds are equal)."""
    per_seed = [scores(total(map_cms(rule, df, images, gt_totals, r11_cms, s), images)) for s in SEEDS]
    return {k: float(np.mean([p[k] for p in per_seed])) for k in per_seed[0]} | \
           {f"destroyed_iou_s{s}": p["destroyed_iou"] for s, p in zip(SEEDS, per_seed)}


rows = []
for name, rule in {**REFERENCES, **CANDIDATES}.items():
    rows.append({"candidate": name, **evaluate(rule, IV, VAL_IMAGES, GT_VAL, R11_VAL)})
VAL = pd.DataFrame(rows)
VAL.round(4).to_csv(OUT / "candidates_validation.csv", index=False)
print(VAL.round(4).to_string(index=False))

cands = VAL[~VAL["candidate"].str.contains("reference")].copy()
best = cands["destroyed_iou"].max()
tied = cands[cands["destroyed_iou"] >= best - TIE].copy()
tied["order"] = tied["candidate"].str[:2].map(ORDER.index)
WINNER = tied.sort_values(["order", "destroyed_iou"], ascending=[True, False]).iloc[0]["candidate"]
print(f"\n✓ The rule picks: {WINNER} (destroyed IoU {VAL.set_index('candidate').loc[WINNER, 'destroyed_iou']:.4f} on the 605 clean pairs; "
      f"{len(tied)} candidate(s) within {TIE} of the best)")

                                           candidate   miou  destroyed_iou  building_iou  destroyed_iou_s42  destroyed_iou_s43  destroyed_iou_s44
                               R11 alone (reference) 0.5805         0.2692        0.7429             0.2692             0.2692             0.2692
                   Find-then-grade alone (reference) 0.5474         0.3068        0.6307             0.3161             0.3063             0.2982
                         C1, R11 grades the outlines 0.5304         0.2139        0.6272             0.2139             0.2139             0.2139
       C2, grader: building or not; R11: which class 0.5307         0.2125        0.6307             0.2127             0.2125             0.2124
                                 C3, blend, w = 0.25 0.5314         0.2135        0.6293             0.2122             0.2138             0.2145
                                  C3, blend, w = 0.5 0.5410         0.2391        0.6314             0.2380             0.23

In [4]:
# ── Cell 4: The winner, once, on the 2,007 clean test pairs, beside R11 and find-then-grade; then resampling ──
rules = {**REFERENCES, **CANDIDATES}
methods = {"R11 alone": rules["R11 alone (reference)"], "Find-then-grade alone": rules["Find-then-grade alone (reference)"],
           WINNER: rules[WINNER]}
turkey = [i for i in TEST_IMAGES if event_of(i).startswith("turkey_earthquake")]
rest = [i for i in TEST_IMAGES if not event_of(i).startswith("turkey_earthquake")]
per_seed_cms = {m: [map_cms(r, IT, TEST_IMAGES, GT_TEST, R11_TEST, s) for s in SEEDS] for m, r in methods.items()}

rows = []
for m, cms_list in per_seed_cms.items():
    row = {"method": m}
    for scope, images in (("all", TEST_IMAGES), ("Turkey", turkey), ("without Turkey", rest)):
        s = [scores(total(cms, images)) for cms in cms_list]
        row[f"destroyed_iou ({scope})"] = float(np.mean([x["destroyed_iou"] for x in s]))
        if scope == "all":
            row["miou"] = float(np.mean([x["miou"] for x in s]))
            row["building_iou"] = float(np.mean([x["building_iou"] for x in s]))
    rows.append(row)
TEST = pd.DataFrame(rows)
TEST.round(4).to_csv(OUT / "test_confirmation.csv", index=False)
print(f"{len(TEST_IMAGES):,} clean test pairs ({len(turkey)} Turkey):")
print(TEST.round(4).to_string(index=False))

# Resampling, as E7 step 2b: the pairs drawn with replacement 10,000 times; destroyed IoU from summed true positives and
# unions; seeds averaged inside each draw
def destroyed_parts(cms):
    tp = np.array([cms[i][3, 3] for i in TEST_IMAGES], np.float64)
    union = np.array([cms[i][3, :].sum() + cms[i][:, 3].sum() - cms[i][3, 3] for i in TEST_IMAGES], np.float64)
    return tp, union


parts = {m: [destroyed_parts(c) for c in cms_list] for m, cms_list in per_seed_cms.items()}
rng = np.random.default_rng(RNG_SEED)
idx = rng.integers(0, len(TEST_IMAGES), size=(N_RESAMPLES, len(TEST_IMAGES)))
counts = np.stack([np.bincount(row, minlength=len(TEST_IMAGES)) for row in idx])        # draws x pairs


def draws(m):
    return np.mean([(counts @ tp) / (counts @ union) for tp, union in parts[m]], axis=0)


res = []
for other in ("R11 alone", "Find-then-grade alone"):
    d = draws(WINNER) - draws(other)
    observed = TEST.set_index("method").loc[WINNER, "destroyed_iou (all)"] - TEST.set_index("method").loc[other, "destroyed_iou (all)"]
    res.append({"comparison": f"{WINNER} minus {other}", "observed": round(float(observed), 4),
                "low_2.5%": round(float(np.percentile(d, 2.5)), 4), "high_97.5%": round(float(np.percentile(d, 97.5)), 4),
                "share_above_0": round(float((d > 0).mean()), 4)})
RES = pd.DataFrame(res)
RES.to_csv(OUT / "resampling.csv", index=False)
print("\nDestroyed IoU differences, the middle 95% of 10,000 redraws of the pairs:")
print(RES.to_string(index=False))

2,007 clean test pairs (414 Turkey):
                                              method  destroyed_iou (all)   miou  building_iou  destroyed_iou (Turkey)  destroyed_iou (without Turkey)
                                           R11 alone               0.3984 0.5721        0.6944                  0.0867                          0.4672
                               Find-then-grade alone               0.3898 0.5129        0.5416                  0.3231                          0.4105
C4, R11 on xBD images, find-then-grade on the others               0.4296 0.5555        0.6221                  0.3231                          0.4610



Destroyed IoU differences, the middle 95% of 10,000 redraws of the pairs:
                                                                      comparison  observed  low_2.5%  high_97.5%  share_above_0
            C4, R11 on xBD images, find-then-grade on the others minus R11 alone    0.0312    0.0075      0.0569         0.9944
C4, R11 on xBD images, find-then-grade on the others minus Find-then-grade alone    0.0397    0.0265      0.0545         1.0000


In [5]:
# ── Cell 5: README ──────────────────────────────────────────────────────────
def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


t = TEST.set_index("method")
lead = {r["comparison"].split(" minus ")[1]: r for r in res}
lines = [
    "# E7 step 2c: option (c), a mix chosen on validation, confirmed once on the test pairs", "",
    "Run on Abrar's PC, no GPU, from saved files only. The question, the candidates, the rule and the expectation are in the "
    "notebook's first cell (`e7-step2c-mix-choice.ipynb`), written before the run.", "",
    "## Check first: the rebuilt scores reproduce the published ones", "", md(CHECKS), "",
    "## The candidates on the 605 clean validation pairs", "",
    "Destroyed IoU is the mean of the three grader seeds (R11 alone and C1 use no grader).", "",
    md(VAL.round(4)), "",
    f"**The rule's choice:** {WINNER}.", "",
    "## Once on the 2,007 clean test pairs", "", md(TEST.round(4)), "",
    "## Is the lead more than noise? (10,000 redraws of the pairs, as E7 step 2b)", "", md(RES), "",
    "## What it means", "",
    f"- On the test pairs, {WINNER} scores destroyed IoU {t.loc[WINNER, 'destroyed_iou (all)']:.4f}, "
    f"against {t.loc['R11 alone', 'destroyed_iou (all)']:.4f} for R11 and {t.loc['Find-then-grade alone', 'destroyed_iou (all)']:.4f} "
    f"for find-then-grade.",
    f"- Its lead over R11: {lead['R11 alone']['observed']:+.4f} (middle 95%: {lead['R11 alone']['low_2.5%']:+.4f} to "
    f"{lead['R11 alone']['high_97.5%']:+.4f}); over find-then-grade: {lead['Find-then-grade alone']['observed']:+.4f} "
    f"({lead['Find-then-grade alone']['low_2.5%']:+.4f} to {lead['Find-then-grade alone']['high_97.5%']:+.4f}).",
    "- This doesn't choose the system's method by itself: Abrar decides, with these numbers.", "",
    "Files: `candidates_validation.csv`, `test_confirmation.csv`, `resampling.csv`.",
]
(OUT / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

# E7 step 2c: option (c), a mix chosen on validation, confirmed once on the test pairs

Run on Abrar's PC, no GPU, from saved files only. The question, the candidates, the rule and the expectation are in the notebook's first cell (`e7-step2c-mix-choice.ipynb`), written before the run.

## Check first: the rebuilt scores reproduce the published ones

| check                                              |   rebuilt |   published | pass   |
|:---------------------------------------------------|----------:|------------:|:-------|
| validation, find-then-grade seed 42: destroyed IoU |    0.3161 |      0.3161 | True   |
| validation, find-then-grade seed 43: destroyed IoU |    0.3063 |      0.3051 | True   |
| validation, find-then-grade seed 44: destroyed IoU |    0.2982 |      0.2982 | True   |
| validation, R11 with TTA: destroyed IoU            |    0.2692 |      0.2692 | True   |
| validation, R11 with TTA: mIoU                     |    0.5805 |      0.5805 | True   |
| test, find-then-